In [ ]:
!pip install contractions spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.8/114.8 kB 7.9 MB/s eta 0:00:00


In [ ]:
!python -m spacy download en_core_web_sm



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 40.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import zipfile
import os
import pandas as pd
import re
import nltk
import spacy
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


In [ ]:

zip_path = "/content/covid19_tweets.csv.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
df = pd.read_csv("/content/dataset/covid19_tweets.csv")

print("Dataset shape:", df.shape)
df.head()



Dataset shape: (179108, 13)


,user_name,user_location,user_description,user_created,user_followers,user_friends,user_favourites,user_verified,date,text,hashtags,source,is_retweet
0,ᏉᎥ☻լꂅϮ,astroworld,wednesday addams as a disney princess keepin i...,2017-05-26 05:46:42,624,950,18775,False,2020-07-25 12:27:21,If I smelled the scent of hand sanitizers toda...,NaN,Twitter for iPhone,False
1,Tom Basile 🇺🇸,"New York, NY","Husband, Father, Columnist & Commentator. Auth...",2009-04-16 20:06:23,2253,1677,24,True,2020-07-25 12:27:17,Hey @Yankees @YankeesPR and @MLB - wouldn't it...,NaN,Twitter for Android,False
2,Time4fisticuffs,"Pewee Valley, KY",#Christian #Catholic #Conservative #Reagan #Re...,2009-02-28 18:57:41,9275,9525,7254,False,2020-07-25 12:27:14,@diane3443 @wdunlap @realDonaldTrump Trump nev...,['COVID19'],Twitter for Android,False
3,ethel mertz,Stuck in the Middle,#Browns #Indians #ClevelandProud #[]_[] #Cavs ...,2019-03-07 01:45:06,197,987,1488,False,2020-07-25 12:27:10,@brookbanktv The one gift #COVID19 has give me...,['COVID19'],Twitter for iPhone,False
4,DIPR-J&K,Jammu and Kashmir,🖊️Official Twitter handle of Department of Inf...,2017-02-12 06:45:15,101009,168,101,False,2020-07-25 12:27:08,25 July : Media Bulletin on Novel #CoronaVirus...,"['CoronaVirusUpdates', 'COVID19']",Twitter for Android,False


In [ ]:
print("Shape:", df.shape)

Shape: (179108, 13)


In [ ]:
print("\nColumns:")
print(df.columns.tolist())


Columns:
['user_name', 'user_location', 'user_description', 'user_created', 'user_followers', 'user_friends', 'user_favourites', 'user_verified', 'date', 'text', 'hashtags', 'source', 'is_retweet']


In [ ]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
user_name               0
user_location       36771
user_description    10286
user_created            0
user_followers          0
user_friends            0
user_favourites         0
user_verified           0
date                    0
text                    0
hashtags            51334
source                 77
is_retweet              0
dtype: int64


In [ ]:
print("\nDuplicate Rows:")
print(df.duplicated().sum())


Duplicate Rows:
0


In [ ]:
tweets = df[['text']].copy()

tweets.head()

,text
0,If I smelled the scent of hand sanitizers toda...
1,Hey @Yankees @YankeesPR and @MLB - wouldn't it...
2,@diane3443 @wdunlap @realDonaldTrump Trump nev...
3,@brookbanktv The one gift #COVID19 has give me...
4,25 July : Media Bulletin on Novel #CoronaVirus...


In [ ]:
tweets.dropna(subset=['text'], inplace=True)
print(tweets.shape)

(179108, 1)


In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

# English stopwords
stop_words = set(stopwords.words('english'))

print("All NLP libraries loaded successfully!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


All NLP libraries loaded successfully!


In [ ]:
def clean_tweet(text):

    text = str(text).lower()

    # remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove @mentions
    text = re.sub(r'@\w+', '', text)

    # remove hashtag symbol but keep word
    text = re.sub(r'#', '', text)

    # remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
tweets['clean_text'] = tweets['text'].apply(clean_tweet)

In [ ]:
tweets[['text', 'clean_text']].head()

,text,clean_text
0,If I smelled the scent of hand sanitizers toda...,if i smelled the scent of hand sanitizers toda...
1,Hey @Yankees @YankeesPR and @MLB - wouldn't it...,hey and wouldn t it have made more sense to ha...
2,@diane3443 @wdunlap @realDonaldTrump Trump nev...,trump never once claimed covid was a hoax we a...
3,@brookbanktv The one gift #COVID19 has give me...,the one gift covid has give me is an appreciat...
4,25 July : Media Bulletin on Novel #CoronaVirus...,july media bulletin on novel coronavirusupdate...


In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):

    words = text.split()

    filtered_words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)

In [ ]:
tweets['stop_removed'] = tweets['clean_text'].apply(remove_stopwords)

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

def stem_text(text):

    words = text.split()

    stemmed_words = [
        stemmer.stem(word)
        for word in words
    ]

    return " ".join(stemmed_words)

In [ ]:
tweets['stemmed_text'] = tweets['stop_removed'].apply(stem_text)

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):

    words = text.split()

    lemmas = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(lemmas)

In [ ]:
tweets['lemmatized_text'] = tweets['stop_removed'].apply(lemmatize_text)

In [ ]:
tweets[
    ['text',
     'clean_text',
     'stop_removed',
     'stemmed_text',
     'lemmatized_text']
].head(10)

,text,clean_text,stop_removed,stemmed_text,lemmatized_text
0,If I smelled the scent of hand sanitizers toda...,if i smelled the scent of hand sanitizers toda...,smelled scent hand sanitizers today someone pa...,smell scent hand sanit today someon past would...,smelled scent hand sanitizers today someone pa...
1,Hey @Yankees @YankeesPR and @MLB - wouldn't it...,hey and wouldn t it have made more sense to ha...,hey made sense players pay respects,hey made sens player pay respect,hey made sense player pay respect
2,@diane3443 @wdunlap @realDonaldTrump Trump nev...,trump never once claimed covid was a hoax we a...,trump never claimed covid hoax claim effort,trump never claim covid hoax claim effort,trump never claimed covid hoax claim effort
3,@brookbanktv The one gift #COVID19 has give me...,the one gift covid has give me is an appreciat...,one gift covid give appreciation simple things...,one gift covid give appreci simpl thing alway ...,one gift covid give appreciation simple thing ...
4,25 July : Media Bulletin on Novel #CoronaVirus...,july media bulletin on novel coronavirusupdate...,july media bulletin novel coronavirusupdates c...,juli media bulletin novel coronavirusupd covid,july medium bulletin novel coronavirusupdates ...
5,#coronavirus #covid19 deaths continue to rise....,coronavirus covid deaths continue to rise it s...,coronavirus covid deaths continue rise almost ...,coronaviru covid death continu rise almost bad...,coronavirus covid death continue rise almost b...
6,How #COVID19 Will Change Work in General (and ...,how covid will change work in general and recr...,covid change work general recruiting specifica...,covid chang work gener recruit specif via recruit,covid change work general recruiting specifica...
7,You now have to wear face coverings when out s...,you now have to wear face coverings when out s...,wear face coverings shopping includes visit lo...,wear face cover shop includ visit local commun...,wear face covering shopping includes visit loc...
8,Praying for good health and recovery of @Chouh...,praying for good health and recovery of covid ...,praying good health recovery covid covidpositive,pray good health recoveri covid covidposit,praying good health recovery covid covidpositive
9,POPE AS GOD - Prophet Sadhu Sundar Selvaraj. W...,pope as god prophet sadhu sundar selvaraj watc...,pope god prophet sadhu sundar selvaraj watch h...,pope god prophet sadhu sundar selvaraj watch h...,pope god prophet sadhu sundar selvaraj watch h...


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 30.2 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec

In [ ]:
sentences = [
    sentence.split()
    for sentence in tweets['lemmatized_text']
]

In [ ]:
w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4
)

In [ ]:
print(len(w2v_model.wv))

18840


In [ ]:
import numpy as np

def tweet_embedding(text):

    words = text.split()

    vectors = [
        w2v_model.wv[word]
        for word in words
        if word in w2v_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [ ]:
tweets['embedding'] = tweets['lemmatized_text'].apply(tweet_embedding)

In [ ]:
print(tweets['embedding'].iloc[0].shape)

(100,)


In [ ]:
tweets.head()

,text,clean_text,stop_removed,stemmed_text,lemmatized_text,embedding
0,If I smelled the scent of hand sanitizers toda...,if i smelled the scent of hand sanitizers toda...,smelled scent hand sanitizers today someone pa...,smell scent hand sanit today someon past would...,smelled scent hand sanitizers today someone pa...,"[0.2823174, -0.16008563, 0.103586406, 0.167927..."
1,Hey @Yankees @YankeesPR and @MLB - wouldn't it...,hey and wouldn t it have made more sense to ha...,hey made sense players pay respects,hey made sens player pay respect,hey made sense player pay respect,"[0.30426022, -0.23753345, 0.1668333, 0.5022762..."
2,@diane3443 @wdunlap @realDonaldTrump Trump nev...,trump never once claimed covid was a hoax we a...,trump never claimed covid hoax claim effort,trump never claim covid hoax claim effort,trump never claimed covid hoax claim effort,"[-0.23105025, -0.16251493, 0.1407795, 0.094348..."
3,@brookbanktv The one gift #COVID19 has give me...,the one gift covid has give me is an appreciat...,one gift covid give appreciation simple things...,one gift covid give appreci simpl thing alway ...,one gift covid give appreciation simple thing ...,"[0.34554183, -0.025163362, 0.198466, -0.102280..."
4,25 July : Media Bulletin on Novel #CoronaVirus...,july media bulletin on novel coronavirusupdate...,july media bulletin novel coronavirusupdates c...,juli media bulletin novel coronavirusupd covid,july medium bulletin novel coronavirusupdates ...,"[-0.124273896, 0.75222784, 0.567009, 0.1684191..."
